# 05 — Model Evaluation

Metrics split into two questions that are often conflated:

* **Ranking** (ROC-AUC, PR-AUC) — can the model order customers by risk?
* **Probability quality** (Brier, ECE, calibration curve) — is a predicted 0.7 actually a 70% chance?

The second matters because the decision layer multiplies probability by customer value. A model that ranks well but is miscalibrated by 2x inflates every expected-value calculation.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
cfg = load_config()

In [2]:
from src.data.loader import make_dataset
from src.features.preprocess import build_pipeline
from xgboost import XGBClassifier

ds = make_dataset(cfg)
pipe = build_pipeline(XGBClassifier(
    random_state=cfg.seed, n_jobs=1, tree_method='hist',
    eval_metric='logloss', n_estimators=300, max_depth=4,
    learning_rate=0.02, subsample=1.0, colsample_bytree=0.6,
    min_child_weight=1, reg_lambda=0.5), scale=False)
pipe.fit(ds.X_train, ds.y_train)
prob = pipe.predict_proba(ds.X_test)[:, 1]

## Why accuracy is not the headline

In [3]:
from src.evaluation.metrics import evaluate, metrics_frame
m = evaluate(ds.y_test, prob, 0.5)
print(f'model accuracy at 0.5        : {m.accuracy:.4f}')
print(f'"nobody churns" accuracy      : {1 - ds.y_test.mean():.4f}')
print(f'...and it finds {0} churners of {int(ds.y_test.sum())}')

model accuracy at 0.5        : 0.7873
"nobody churns" accuracy      : 0.7354
...and it finds 0 churners of 372


## Calibration

Calibrators are fitted with internal CV on the **training** split and judged on the untouched test set. The uncalibrated model competes as a candidate: a booster trained on logloss is often already calibrated, and a calibrator that does not measurably help is a component to maintain for nothing.

In [4]:
from src.evaluation import metrics as M
best_cal, all_cal = M.calibrate(pipe, ds, cfg)
import pandas as pd
pd.DataFrame([{'method': c.method, 'brier': c.brier_after,
               'ece': c.ece_after, 'log_loss': c.log_loss_after,
               'pr_auc': c.pr_auc_after} for c in all_cal]).round(4)

,method,brier,ece,log_loss,pr_auc
0,none,0.1425,0.0257,0.4379,0.6134
1,sigmoid,0.1437,0.0461,0.4428,0.6178
2,isotonic,0.1421,0.0260,0.4384,0.6185


In [5]:
print('selected:', best_cal.method)
print('parsimony guard (min Brier gain):', cfg['calibration']['min_brier_gain'])

selected: none
parsimony guard (min Brier gain): 0.001


## Split stability

A single 80/20 split of ~7k rows is a noisy estimate. Reporting one number from one split invites two errors: mistaking split luck for model quality, and quietly re-drawing the split until the number improves.

In [6]:
stability = M.split_stability(pipe, cfg)
stability.round(4)

,fold,is_designated_test,n,churn_rate,pr_auc,roc_auc,brier
0,0,True,1406,0.2646,0.6134,0.8270,0.1425
1,1,False,1405,0.2648,0.6868,0.8589,0.1285
2,2,False,1406,0.2646,0.6436,0.8471,0.1368
3,3,False,1406,0.2646,0.6855,0.8552,0.1309
4,4,False,1404,0.2642,0.6752,0.8534,0.1310


In [7]:
print(f"designated test fold : {stability.loc[0,'pr_auc']:.4f}")
print(f"across-fold mean     : {stability['pr_auc'].mean():.4f} "
      f"(sd {stability['pr_auc'].std():.4f})")

designated test fold : 0.6134
across-fold mean     : 0.6609 (sd 0.0318)


## Threshold selection on a cost/value framework

> The economic parameters are **assumptions** declared in `configs/config.yaml`. The dataset has no campaign costs, offer values or intervention outcomes, so they cannot be estimated from it. Figures are in neutral currency units (CU).

In [8]:
from src.evaluation.threshold import Economics, optimise, sensitivity_analysis
econ = Economics.from_config(cfg)
value = econ.customer_value(ds.X_test)
res = optimise(ds.y_test.to_numpy(), prob, value, econ)
print(f'best global threshold : {res.best_threshold:.2f} -> {res.best_net_value:,.0f} CU')
print(f'contact everyone      : {res.baseline_contact_all:,.0f} CU')
print(f'contact nobody        : 0 CU')
print(f'per-customer EV rule  : {res.per_customer_net_value:,.0f} CU '
      f'({res.per_customer_contacts} contacts)')

best global threshold : 0.13 -> 63,446 CU
contact everyone      : 49,916 CU
contact nobody        : 0 CU
per-customer EV rule  : 63,799 CU (713 contacts)


The per-customer rule wins because the break-even probability varies with customer value. A single global cut cannot express that a high-value customer is worth contacting at much lower risk.

In [9]:
be = econ.breakeven_probability(value)
print(f'break-even probability ranges {be.min():.1%} to {be.max():.1%}')

break-even probability ranges 8.2% to 51.9%


In [10]:
metrics_frame({'threshold_0.50': evaluate(ds.y_test, prob, 0.5),
               f'threshold_{res.best_threshold:.2f}':
                   evaluate(ds.y_test, prob, res.best_threshold)}).round(4)

,threshold,roc_auc,pr_auc,precision,recall,f1,accuracy,brier,log_loss,tn,fp,fn,tp
threshold_0.50,0.50,0.827,0.6134,0.6308,0.4731,0.5407,0.7873,0.1425,0.4379,931.0,103.0,196.0,176.0
threshold_0.13,0.13,0.827,0.6134,0.4186,0.9059,0.5726,0.6422,0.1425,0.4379,566.0,468.0,35.0,337.0


## Sensitivity: the threshold is a function of the assumptions

In [11]:
sensitivity_analysis(ds.y_test.to_numpy(), prob, value, cfg).round(3)

,success_rate,incentive_cost,best_threshold,net_value,contact_rate,precision,recall,per_customer_rule_net_value
0,0.1,10.0,0.18,18726.60,0.496,0.456,0.855,19270.92
1,0.1,30.0,0.39,8383.12,0.284,0.598,0.642,9001.22
2,0.1,60.0,0.75,1148.94,0.051,0.819,0.159,1838.78
3,0.2,10.0,0.05,49398.24,0.719,0.357,0.970,49663.56
4,0.2,30.0,0.18,33968.20,0.496,0.456,0.855,35485.24
5,0.2,60.0,0.39,18766.24,0.284,0.598,0.642,20415.44
6,0.3,10.0,0.03,81711.12,0.773,0.335,0.978,81874.32
7,0.3,30.0,0.13,63445.54,0.573,0.419,0.906,63798.86
8,0.3,60.0,0.23,43019.32,0.434,0.490,0.804,46583.42
9,0.4,10.0,0.03,114383.16,0.773,0.335,0.978,114891.72


## Error analysis and segment-level performance

In [12]:
from src.evaluation.error_analysis import (error_frame, error_costs,
                                           profile_errors, hardest_cases)
errors = error_frame(ds.X_test, ds.y_test.to_numpy(), prob, res.best_threshold)
profile_errors(errors).round(2)

,feature,true_positive,false_negative,false_positive,true_negative
0,tenure,16.19,44.17,25.18,48.08
1,MonthlyCharges,75.52,59.57,70.45,53.24
2,TotalCharges,1460.38,3116.47,2143.42,2894.65
3,predicted probability,0.52,0.07,0.35,0.04


In [13]:
costs = error_costs(errors, cfg)
for k, v in costs.items():
    print(f'{k:28s} {v:,.2f}' if isinstance(v, float) else f'{k:28s} {v}')

n_false_negative             35
n_false_positive             468
fn_opportunity_cost_cu       7,505.82
fp_wasted_spend_cu           16,380.00
mean_fn_cost_cu              214.45
mean_fp_cost_cu              35.00
cost_ratio_fn_to_fp          6.13


In [14]:
hardest_cases(errors)['confident_false_negatives']

,probability,tenure,Contract,MonthlyCharges,InternetService,PaymentMethod,TechSupport
0,0.008131,59,Two year,19.35,No,Mailed check,No internet service
1,0.013259,56,Two year,25.15,No,Credit card (automatic),No internet service
2,0.013692,55,Two year,57.55,DSL,Bank transfer (automatic),Yes
3,0.014321,65,Two year,74.80,DSL,Bank transfer (automatic),Yes
4,0.015726,64,Two year,70.20,DSL,Credit card (automatic),Yes


In [15]:
from src.evaluation.segments import performance_disparities
disp, spreads = performance_disparities(
    ds.X_test, ds.y_test.to_numpy(), prob, res.best_threshold)
disp.round(3)

,attribute,segment,n,churn_rate,roc_auc,pr_auc,brier,mean_predicted,calibration_gap,recall,precision,flag_rate
0,Contract,Month-to-month,792,0.409,0.741,0.646,0.201,0.415,0.006,0.969,0.441,0.899
1,Contract,Two year,355,0.039,0.699,0.108,0.037,0.030,-0.009,0.143,0.143,0.039
2,Contract,One year,259,0.131,0.714,0.273,0.107,0.101,-0.030,0.618,0.266,0.305
3,tenure_band,49-72m,464,0.106,0.771,0.340,0.084,0.089,-0.016,0.612,0.254,0.254
4,tenure_band,25-48m,313,0.208,0.778,0.447,0.140,0.225,0.017,0.892,0.320,0.578
5,tenure_band,0-6m,311,0.511,0.754,0.726,0.201,0.516,0.005,0.994,0.532,0.955
6,tenure_band,13-24m,191,0.267,0.758,0.468,0.168,0.273,0.006,0.882,0.372,0.634
7,tenure_band,7-12m,127,0.378,0.794,0.659,0.182,0.328,-0.050,0.958,0.523,0.693
8,InternetService,Fiber optic,617,0.415,0.778,0.682,0.189,0.408,-0.007,0.965,0.494,0.810
9,InternetService,DSL,469,0.192,0.780,0.421,0.132,0.189,-0.002,0.833,0.316,0.505


In [16]:
sorted(spreads.items(), key=lambda kv: -kv[1])[:6]

[('Contract_recall_spread', 0.8262786596119929),
 ('InternetService_recall_spread', 0.38792067307692313),
 ('tenure_band_recall_spread', 0.3814657938647157),
 ('PaymentMethod_recall_spread', 0.19249018324607325),
 ('PaymentMethod_roc_auc_spread', 0.11778383913860802),
 ('SeniorCitizen_recall_spread', 0.06723602484472058)]

Read these as **stability and coverage checks**, not a fairness certification. A large spread on `gender` would be a warning sign; a small one is not evidence of fairness in any broader sense.